# 03 — Scanning Kernels

Covers Phase 3 of the kernel roadmap:
- `prefix_sum` — parallel cumulative sum (parallel prefix / scan)
- `cummax` — running maximum

**Metric**: GB/s — `(2 × n × bytes × 1e-9) / (ms × 1e-3)` (read + write)

In [ ]:
# ── Setup: mount Drive and clone / pull the repo ─────────────────────────────
import os
from google.colab import drive

drive.mount("/content/drive")

REPO_URL    = "https://github.com/Bhavikupadhyay/triton-kernels.git"
REPO_BRANCH = "feature/prefix-sum"
REPO_DIR    = "/content/drive/MyDrive/triton-kernels"

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} fetch --all
    !git -C {REPO_DIR} checkout -f {REPO_BRANCH}
    !git -C {REPO_DIR} reset --hard origin/{REPO_BRANCH}
else:
    !git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!bash scripts/setup_colab.sh

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import torch
import triton

from kernels.scanning.prefix_sum import prefix_sum, test_prefix_sum, benchmark_prefix_sum

# Uncomment as kernels are implemented:
# from kernels.scanning.cummax import cummax, test_cummax, benchmark_cummax

print("Imports ready")

## 1. prefix_sum

**File**: `kernels/scanning/prefix_sum.py`  
**PyTorch equivalent**: `torch.cumsum(x, dim=0)`  
**Algorithm**: Parallel prefix scan (Hillis-Steele or Blelloch)

In [ ]:
# ── prefix_sum: Correctness ───────────────────────────────────────────────────
test_prefix_sum()

In [ ]:
# ── prefix_sum: Benchmark ─────────────────────────────────────────────────────
import os
os.makedirs("benchmarks/results/scanning", exist_ok=True)

benchmark_prefix_sum.run(
    print_data=True,
    show_plots=True,
    save_path="benchmarks/results/scanning",
)

**Interpretation — prefix_sum (T4, fp32)**

**PyTorch wins at every size except n=1K, and the gap doubles at large n.** Triton peaks at 158 GB/s (n=512K) then drops to a plateau of ~115 GB/s. PyTorch keeps climbing to ~231 GB/s.

**n=1K — Triton faster (1.08 vs 0.67 GB/s).** Both are launch-latency dominated at this size. PyTorch has more overhead per launch.

**n=2K–256K — PyTorch ahead by 3–7%, both climbing.** Roughly tied at n=128K–256K (~85, ~128 GB/s). The three-pass overhead hasn't made itself visible yet.

**n=512K — Triton peaks at 158 GB/s, then drops.** This is the T4's L2 cache boundary. The intermediate `out` buffer is 2 MB at n=512K — fits inside T4's 4 MB L2, so pass 3 reads the local scans from cache rather than HBM. That cache-hit boost is what inflates the n=512K number.

**n≥2M — Triton plateaus at ~113–115 GB/s; PyTorch reaches ~231 GB/s.** Once the `out` buffer exceeds L2 (4 MB ≈ 1M fp32 elements), pass 3 reads cold from HBM every time. The full 4n HBM cost is now visible: pass 1 reads x and writes local scans (2n), pass 3 reads and rewrites them (2n). Total: 4n. Our formula reports 2n/time, so we show half the rate of PyTorch, which does a genuine 2n (read once, write once).

**Both kernels hit the same effective HBM throughput.** Triton's ~115 GB/s × (4n / 2n) = ~230 GB/s actual. PyTorch's ~231 GB/s × (2n / 2n) = ~231 GB/s actual. The hardware is equally saturated — we're just doing twice the work per output element.

The limitation is algorithmic: three separate kernel launches communicating through global memory can't avoid the double read/write of the intermediate buffer. PyTorch uses CUB's single-pass shared-memory implementation, which keeps intra-block intermediate results in SRAM and only touches HBM once per element each way.

## 2. cummax

**File**: `kernels/scanning/cummax.py`  
**PyTorch equivalent**: `torch.cummax(x, dim=0)`

In [ ]:
# ── cummax: Correctness ──────────────────────────────────────────────────────
# test_cummax()

In [ ]:
# ── cummax: Benchmark ────────────────────────────────────────────────────────
# import os; os.makedirs("benchmarks/results/scanning", exist_ok=True)
# benchmark_cummax.run(print_data=True, show_plots=True,
#     save_path="benchmarks/results/scanning")

**Interpretation**: Add notes here.

In [ ]:
# ── Summary Table ────────────────────────────────────────────────────────────
# import pandas as pd, glob
# csvs = glob.glob("benchmarks/results/scanning/*.csv")
# if csvs:
#     print(pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True).to_string(index=False))
# else:
#     print("No CSVs yet.")